# 2. Data Understanding (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style='whitegrid')

## 2.1 Load Dataset

In [ ]:
df = pd.read_csv('../Datasets/hypertension_dataset.csv')
display(df.head())
print('\nInfo Dataset:')
df.info()
print('\nStatistik Deskriptif:')
display(df.describe(include='all'))

## 2.2 Distribusi Variabel Target (Has_Hypertension)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Has_Hypertension', palette='Set2')
plt.title('Distribusi Kelas Has_Hypertension')
plt.show()

target_counts = df['Has_Hypertension'].value_counts()
target_pct = df['Has_Hypertension'].value_counts(normalize=True) * 100

print("Jumlah per kelas:")
print(target_counts)
print("\nPersentase per kelas:")
print(target_pct.round(2))

imbalance_ratio = target_counts.max() / target_counts.min()
print(f"\nRasio imbalance: {imbalance_ratio:.2f} : 1")

## 2.3 Analisis Numerik: Salt Intake vs Hypertension

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Has_Hypertension', y='Salt_Intake', palette='Set2')
plt.title('Distribusi Konsumsi Garam Berdasarkan Status Hipertensi')
plt.show()

## 2.4 Cek Outlier

In [ ]:
num_cols_check = df.select_dtypes(include=['number']).columns.tolist()

print("=== Hasil Deteksi Outlier ===\n")
for col in num_cols_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    outlier = df[(df[col] < batas_bawah) | (df[col] > batas_atas)]
    pct = len(outlier) / len(df) * 100

    print(f"{col}:")
    print(f"  Batas bawah : {batas_bawah:.2f}")
    print(f"  Batas atas  : {batas_atas:.2f}")
    print(f"  Jumlah outlier: {len(outlier)} ({pct:.2f}% dari total data)\n")

## 2.5 Korelasi Antar Variabel Numerik

In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Numerik')
plt.show()

## 2.6 Distribusi Fitur Kategorikal

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.drop('Has_Hypertension')
plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_cols, 1):
    plt.subplot(2, 3, i)
    sns.countplot(data=df, x=col, hue='Has_Hypertension', palette='Set2')
    plt.xticks(rotation=45)
    plt.title(f'{col} vs Hypertension')
plt.tight_layout()
plt.show()

## 2.7 Tabel Acuan Indikasi Hipertensi

In [ ]:
# ---------- PLOT Agregasi Rata-rata Fitur Numerik ----------
agg_numeric = df.groupby('Has_Hypertension')[numeric_cols].mean().T

fig, ax = plt.subplots(figsize=(10, 6))
agg_numeric.plot(kind='bar', ax=ax, color=['#81b29a', '#e07a5f'])
ax.set_title('Agregasi: Rata-rata Fitur Numerik (Hipertensi vs Tidak)')
ax.set_ylabel('Nilai Rata-rata')
ax.set_xlabel('Fitur')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Has_Hypertension')

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.tight_layout()
plt.show()

# ---------- TABEL ACUAN GABUNGAN ----------
acuan_rows = []

for col in categorical_cols:
    crosstab_pct = pd.crosstab(df[col], df['Has_Hypertension'], normalize='index') * 100
    for kategori in crosstab_pct.index:
        pct_yes = crosstab_pct.loc[kategori, 'Yes']
        status = "Terindikasi" if pct_yes > 50 else "Tidak Terindikasi"
        selisih = abs(pct_yes - 50)
        acuan_rows.append({
            'Label': f"{col}: {kategori}",
            'Nilai (%)': pct_yes,
            'Status': status,
            'Kekuatan Indikasi': round(selisih, 1)
        })

for col in numeric_cols:
    mean_yes = df[df['Has_Hypertension'] == 'Yes'][col].mean()
    mean_no = df[df['Has_Hypertension'] == 'No'][col].mean()
    threshold = (mean_yes + mean_no) / 2
    arah = ">" if mean_yes > mean_no else "<"
    selisih_relatif = abs(mean_yes - mean_no) / ((mean_yes + mean_no) / 2) * 100
    acuan_rows.append({
        'Label': f"{col} {arah} {threshold:.2f}",
        'Nilai (%)': selisih_relatif,
        'Status': "Terindikasi",
        'Kekuatan Indikasi': round(selisih_relatif, 1)
    })

df_acuan = pd.DataFrame(acuan_rows).sort_values('Kekuatan Indikasi', ascending=False).reset_index(drop=True)

print("\nTabel Acuan Indikasi Hipertensi (Semua Fitur/Kategori):")
display(df_acuan)